# A 同学 · 记忆区网格 mainN40（由 B 的 math.ipynb 裁剪）
n_train=40、30000 步、tag=`mainN40`，只跑距离下标 0/1/2（75 格）。

**已删除**：selfcheck、pilot 4×4、pilot2 5×5、pilot_iters10k（会改写 `DISTANCES`/`CAPACITIES`，或往结果 csv 塞别的 tag）。

路径兼容三种环境：Colab（Drive）、Kaggle（/kaggle/working）、本地。


## 1. 导入

In [ ]:
import os, time, csv, math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("使用设备：", DEVICE)

使用设备： cuda


## 2. 路径（Colab 写 Drive；Kaggle 自动落 /kaggle/working）

In [ ]:
WHO = "A"
OUT_DIR = "/content/drive/MyDrive/diffusion_project_A"
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("没有 Colab/Drive，改用本地目录：", e)
    OUT_DIR = "/kaggle/working/diffusion_project_A" if os.path.exists("/kaggle/working") \
        else "./diffusion_project_A"
os.makedirs(OUT_DIR, exist_ok=True)
RESULT_CSV = os.path.join(OUT_DIR, f"results_{WHO}.csv")
print("输出目录:", OUT_DIR)
print("结果文件:", RESULT_CSV)


Mounted at /content/drive
输出目录: /content/drive/MyDrive/diffusion_project_A
结果文件: /content/drive/MyDrive/diffusion_project_A/results_A.csv


## 3. 参数：主网格 CFG + 记忆区 cfg_mem

In [ ]:
CFG = dict(
    K=8, sigma=0.3, n_train=1000, T=100,
    iters=10000, batch=256, lr=1e-3,
    n_sample=2000, n_traj=10, tdim=32,
)
DISTANCES  = [1.0, 1.5, 2.0, 3.0, 4.5]            # 环半径
CAPACITIES = [(1, 8), (2, 16), (3, 64), (4, 256), (5, 512)]
SEEDS      = [0, 1, 2, 3, 4]

# ---- 记忆区网格：只改训练集大小和步数 ----
cfg_mem = dict(CFG, n_train=40, iters=30000)
TAG_MEM = "mainN40"


## 4. 数据生成、模型、训练、采样、调度（B 原文，不要改）

In [ ]:
##### 圆周等距均值的二维高斯混合 #####
def make_gmm(n, K, radius, sigma, seed):
    """返回 n 个二维点、每个点所属分量编号、K 个均值。radius 越大，模式离得越远。"""
    rng = np.random.default_rng(seed)
    ang = 2 * np.pi * np.arange(K) / K
    means = radius * np.stack([np.cos(ang), np.sin(ang)], axis=1)
    comp = rng.integers(0, K, size=n)
    x = means[comp] + sigma * rng.standard_normal((n, 2))
    return x.astype(np.float32), comp, means

In [ ]:
class Denoiser(nn.Module):
    """MLP 去噪网络：输入 (带噪点, 时间步)，输出预测的噪声。depth=隐藏层数，width=每层宽度。"""
    def __init__(self, depth, width, tdim=32):
        super().__init__()
        self.tdim = tdim
        layers, d_in = [], 2 + tdim
        for i in range(depth):
            layers += [nn.Linear(d_in if i == 0 else width, width), nn.SiLU()]
        layers.append(nn.Linear(width, 2))
        self.net = nn.Sequential(*layers)

    def temb(self, t):
        half = self.tdim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        a = t.float()[:, None] * freqs[None]
        return torch.cat([a.sin(), a.cos()], dim=1)

    def forward(self, x, t):
        return self.net(torch.cat([x, self.temb(t)], dim=1))

In [ ]:
class DDPM:
    """线性噪声日程；beta 按步数缩放，保证无论 T 多大，最后一步都接近纯噪声。"""
    def __init__(self, T, device):
        self.T = T
        s = 1000.0 / T
        self.betas = torch.linspace(1e-4 * s, 0.02 * s, T, device=device)
        self.alphas = 1 - self.betas
        self.abar = torch.cumprod(self.alphas, dim=0)

    def loss(self, model, x0):
        t = torch.randint(0, self.T, (x0.shape[0],), device=x0.device)
        eps = torch.randn_like(x0)
        ab = self.abar[t][:, None]
        xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps
        return ((model(xt, t) - eps) ** 2).mean()


def build_model(depth, width, cfg):
    model = Denoiser(depth, width, cfg["tdim"]).to(DEVICE)
    model.ddpm = DDPM(cfg["T"], DEVICE)   # 把扩散参数挂在模型上，采样时直接用
    return model


def n_params(model):
    return sum(p.numel() for p in model.parameters())


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
def train(depth, width, radius, seed, cfg, ckpt_path=None):
    """训练一个模型。每 1000 步存断点，断线后重跑会从断点继续。"""
    set_seed(seed)
    x_np, _, _ = make_gmm(cfg["n_train"], cfg["K"], radius, cfg["sigma"], seed)
    x = torch.tensor(x_np, device=DEVICE)
    model = build_model(depth, width, cfg)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    start, losses = 0, []
    if ckpt_path and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        start, losses = ck["it"], ck["losses"]
        print(f"  从断点继续：第 {start} 步")
    model.train()
    bs = min(cfg["batch"], len(x))
    for it in range(start, cfg["iters"]):
        idx = torch.randint(0, len(x), (bs,), device=DEVICE)
        loss = model.ddpm.loss(model, x[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if ckpt_path and (it + 1) % 1000 == 0:
            torch.save(dict(model=model.state_dict(), opt=opt.state_dict(),
                            it=it + 1, losses=losses), ckpt_path)
    return model, losses, x_np


@torch.no_grad()
def sample(model, n, return_trajectory=True, seed=0, n_traj=10):
    """
    和 A 约定的采样接口：
      final = sample(model, n, return_trajectory=False)            -> (n, 2)
      final, traj = sample(model, n, return_trajectory=True)       -> traj 形状 (n_traj, n, 2)
    traj[0] 是纯噪声，traj[-1] 就是最终样本，中间是反向扩散过程中等间隔的快照。
    """
    model.eval()
    d = model.ddpm
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    x = torch.randn(n, 2, device=DEVICE, generator=g)
    keep = set(np.linspace(d.T - 1, 0, n_traj - 1).round().astype(int).tolist())
    traj = [x.cpu().numpy()]
    for t in range(d.T - 1, -1, -1):
        tt = torch.full((n,), t, device=DEVICE, dtype=torch.long)
        eps = model(x, tt)
        a, ab, b = d.alphas[t], d.abar[t], d.betas[t]
        mean = (x - b / (1 - ab).sqrt() * eps) / a.sqrt()
        if t > 0:
            x = mean + b.sqrt() * torch.randn(x.shape, device=DEVICE, generator=g)
        else:
            x = mean
        if t in keep:
            traj.append(x.cpu().numpy())
    final = x.cpu().numpy()
    if return_trajectory:
        return final, np.stack(traj)
    return final

In [ ]:
##### 结果文件：一行一次运行。后面三列留空，由 A 的评估代码回填。
FIELDS = ["tag", "dist_idx", "cap_idx", "radius", "depth", "width", "params", "seed",
          "n_train", "T", "K", "final_loss", "seconds", "npz_path",
          "mode_coverage", "mmd", "nn_ratio"]

def append_row(row):
    new = not os.path.exists(RESULT_CSV)
    with open(RESULT_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow({k: row.get(k, "") for k in FIELDS})


def run_one(di, ci, seed, cfg=CFG, tag="main"):
    """跑一个配置（距离档 di、容量档 ci、种子 seed）。已完成的自动跳过。"""
    radius = DISTANCES[di]
    depth, width = CAPACITIES[ci]
    name = f"{tag}_d{di}_c{ci}_s{seed}"
    folder = os.path.join(OUT_DIR, tag)
    os.makedirs(folder, exist_ok=True)
    npz = os.path.join(folder, name + ".npz")
    ckpt = os.path.join(folder, name + ".ckpt")
    if os.path.exists(npz):
        print("跳过（已完成）：", name)
        return
    t0 = time.time()
    model, losses, x_train = train(depth, width, radius, seed, cfg, ckpt)
    final, traj = sample(model, cfg["n_sample"], True, seed=seed, n_traj=cfg["n_traj"])
    np.savez_compressed(npz, final=final, traj=traj, train=x_train,
                        losses=np.array(losses), means_radius=radius)
    torch.save(model.state_dict(), os.path.join(folder, name + ".pt"))
    sec = time.time() - t0
    append_row(dict(tag=tag, dist_idx=di, cap_idx=ci, radius=radius, depth=depth, width=width,
                    params=n_params(model), seed=seed, n_train=cfg["n_train"], T=cfg["T"],
                    K=cfg["K"], final_loss=round(float(np.mean(losses[-200:])), 5),
                    seconds=round(sec, 1), npz_path=npz))
    if os.path.exists(ckpt):
        os.remove(ckpt)
    print(f"完成 {name}：参数量 {n_params(model)}，用时 {sec:.1f}s")


def run_grid(dist_idx_list, cap_idx_list=range(5), seeds=SEEDS, cfg=CFG, tag="main"):
    """批量跑网格。种子放最外层：先把所有格子各跑1个种子，万一时间不够，每格至少有数据。"""
    for s in seeds:
        for di in dist_idx_list:
            for ci in cap_idx_list:
                run_one(di, ci, s, cfg, tag)


def run_ablation(key, values, di=2, ci=2, seeds=(0, 1)):
    """消融：固定一个中间配置，只改 cfg 里的一项（如 n_train、T、K）。"""
    for v in values:
        cfg = dict(CFG); cfg[key] = v
        for s in seeds:
            run_one(di, ci, s, cfg, tag=f"abl_{key}{v}")

## 5. 开跑前核对

In [ ]:
assert cfg_mem["n_train"] == 40 and cfg_mem["iters"] == 30000
assert DISTANCES == [1.0, 1.5, 2.0, 3.0, 4.5]
assert CAPACITIES == [(1, 8), (2, 16), (3, 64), (4, 256), (5, 512)]
print("DEVICE:", DEVICE, "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "无 GPU —— CPU 跑 75×30k 步要 5 小时以上")
print("本次:", 3 * 5 * 5, "格 × 30000 步，tag =", TAG_MEM)


DEVICE: cuda | Tesla T4
本次: 75 格 × 30000 步，tag = mainN40


## 6. 试水一格，记下用时

In [ ]:
run_one(0, 0, 0, cfg=cfg_mem, tag=TAG_MEM)   # 先跑一格，记下用时；30000 步约 3 倍于 10000 步


完成 mainN40_d0_c0_s0：参数量 298，用时 61.8s


## 7. 正式跑 75 格

In [ ]:
run_grid([0, 1, 2], seeds=[0, 1, 2, 3, 4], cfg=cfg_mem, tag=TAG_MEM)


跳过（已完成）： mainN40_d0_c0_s0
完成 mainN40_d0_c1_s0：参数量 866，用时 60.8s
完成 mainN40_d0_c2_s0：参数量 10690，用时 66.8s
完成 mainN40_d0_c3_s0：参数量 206850，用时 76.7s
完成 mainN40_d0_c4_s0：参数量 1069570，用时 85.5s
完成 mainN40_d1_c0_s0：参数量 298，用时 55.4s
完成 mainN40_d1_c1_s0：参数量 866，用时 61.1s
完成 mainN40_d1_c2_s0：参数量 10690，用时 67.6s
完成 mainN40_d1_c3_s0：参数量 206850，用时 76.2s
完成 mainN40_d1_c4_s0：参数量 1069570，用时 85.9s
完成 mainN40_d2_c0_s0：参数量 298，用时 54.9s
完成 mainN40_d2_c1_s0：参数量 866，用时 62.8s
完成 mainN40_d2_c2_s0：参数量 10690，用时 69.5s
完成 mainN40_d2_c3_s0：参数量 206850，用时 78.3s
完成 mainN40_d2_c4_s0：参数量 1069570，用时 87.9s
完成 mainN40_d0_c0_s1：参数量 298，用时 56.4s
完成 mainN40_d0_c1_s1：参数量 866，用时 63.7s
完成 mainN40_d0_c2_s1：参数量 10690，用时 69.5s
完成 mainN40_d0_c3_s1：参数量 206850，用时 77.0s
完成 mainN40_d0_c4_s1：参数量 1069570，用时 87.8s
完成 mainN40_d1_c0_s1：参数量 298，用时 55.4s
完成 mainN40_d1_c1_s1：参数量 866，用时 61.8s
完成 mainN40_d1_c2_s1：参数量 10690，用时 67.2s
完成 mainN40_d1_c3_s1：参数量 206850，用时 74.8s
完成 mainN40_d1_c4_s1：参数量 1069570，用时 86.0s
完成 mainN40_d2_c0_s1：参数量 298，用时 58.3s
完成 m

## 8. 跑完自检

In [ ]:
import pandas as pd
df = pd.read_csv(RESULT_CSV)
df = df[df.tag == TAG_MEM]
print("行数:", len(df), "（应为 75）")
print(df.groupby("dist_idx").size().to_string())
print(df.groupby(["dist_idx", "cap_idx"]).size().unstack().to_string())
print("用时合计(min):", round(df.seconds.sum() / 60, 1))
print("单次中位(s):", df.seconds.median())
